# Nettoyage des fichiers LaDe

Ce notebook traite tous les CSV des dossiers `delivery` et `pickup` de `data/raw`, puis écrit les fichiers nettoyés dans `data/processed` en conservant leurs sous-dossiers. Les fichiers sources restent inchangés. Les résultats sont triés par `accept_time`, du plus récent au plus ancien.

Pour les collectes, la consigne cite `current_gps_*`, qui n'existe pas dans ces CSV. Le jeu de données utilise `accept_gps_*` : une ligne est conservée si au moins une des quatre coordonnées GPS d'acceptation ou de collecte est renseignée. Cela correspond aussi à la règle explicite de suppression lorsque les quatre coordonnées sont vides.

In [1]:
from pathlib import Path
import re
import pandas as pd

# Le notebook peut être lancé depuis la racine du dépôt ou depuis data/raw.
cwd = Path.cwd().resolve()
if (cwd / "data" / "raw").is_dir():
    project_root = cwd
elif cwd.name == "raw" and cwd.parent.name == "data":
    project_root = cwd.parent.parent
else:
    raise RuntimeError("Lancez ce notebook depuis la racine du dépôt ou depuis data/raw.")

raw_dir = project_root / "data" / "raw"
processed_dir = project_root / "data" / "processed"
processed_dir.mkdir(parents=True, exist_ok=True)

columns_to_remove = ["city", "aoi_type", "ds"]
gps_columns = {
    "delivery": ["accept_gps_lng", "accept_gps_lat", "delivery_gps_lng", "delivery_gps_lat"],
    "pickup": ["accept_gps_lng", "accept_gps_lat", "pickup_gps_lng", "pickup_gps_lat"],
}
time_columns_to_remove = {
    "delivery": ["delivery_gps_time", "accept_gps_time"],
    "pickup": ["pickup_gps_time", "accept_gps_time"],
}

print(f"Dossier source : {raw_dir}")
print(f"Dossier destination : {processed_dir}")

Dossier source : /home/yourix/Documents/PROJECTS/IOT/delivery-car/data/raw
Dossier destination : /home/yourix/Documents/PROJECTS/IOT/delivery-car/data/processed


In [2]:
def add_missing_year(value):
    """Ajoute 2022 aux dates MM-DD HH:MM:SS sans année."""
    if pd.isna(value):
        return pd.NA
    value = str(value).strip()
    if re.fullmatch(r"\d{2}-\d{2} \d{2}:\d{2}:\d{2}", value):
        return "2022-" + value
    return value


def clean_file(csv_path, dataset):
    df = pd.read_csv(csv_path)
    rows_read = len(df)

    # Uniformise les cellules vides, y compris celles ne contenant que des espaces.
    df = df.replace(r"^\s*$", pd.NA, regex=True)

    # Retire les lignes entièrement vides en considérant toutes les colonnes sources.
    before = len(df)
    df = df.dropna(how="all")
    empty_rows_removed = before - len(df)

    # Garde les lignes ayant au moins une coordonnée GPS utile.
    available_gps_columns = [col for col in gps_columns[dataset] if col in df.columns]
    if not available_gps_columns:
        raise ValueError(f"Aucune colonne GPS attendue dans {csv_path}")
    before = len(df)
    df = df.dropna(subset=available_gps_columns, how="all")
    gps_rows_removed = before - len(df)

    # Supprime les champs demandés, lorsqu'ils sont présents.
    to_drop = columns_to_remove + time_columns_to_remove[dataset]
    df = df.drop(columns=to_drop, errors="ignore")

    # Les horodatages sont au format MM-DD HH:MM:SS et ne contiennent pas d'année.
    for column in df.columns:
        if "time" in column.lower():
            df[column] = df[column].map(add_missing_year)

    # Trie les tâches par date d'acceptation, de la plus récente à la plus ancienne.
    if "accept_time" not in df.columns:
        raise ValueError(f"Colonne accept_time absente dans {csv_path}")
    df["_accept_time_sort"] = pd.to_datetime(
        df["accept_time"], format="%Y-%m-%d %H:%M:%S", errors="coerce"
    )
    df = (
        df.sort_values("_accept_time_sort", ascending=False, na_position="last", kind="stable")
          .drop(columns="_accept_time_sort")
    )

    destination = processed_dir / csv_path.relative_to(raw_dir)
    destination.parent.mkdir(parents=True, exist_ok=True)
    df.to_csv(destination, index=False)

    return {
        "fichier": str(csv_path.relative_to(raw_dir)),
        "lignes_lues": rows_read,
        "lignes_vides_supprimees": empty_rows_removed,
        "lignes_sans_GPS_supprimees": gps_rows_removed,
        "lignes_ecrites": len(df),
        "destination": str(destination.relative_to(project_root)),
    }

In [3]:
# Exécute le nettoyage sur chaque CSV des deux catégories, y compris les éventuels sous-dossiers.
results = []
for dataset in ("delivery", "pickup"):
    source_dir = raw_dir / dataset
    if not source_dir.is_dir():
        raise FileNotFoundError(f"Dossier absent : {source_dir}")
    csv_files = sorted(source_dir.rglob("*.csv"))
    if not csv_files:
        raise FileNotFoundError(f"Aucun CSV trouvé dans {source_dir}")
    for csv_path in csv_files:
        results.append(clean_file(csv_path, dataset))

summary = pd.DataFrame(results)
summary

,fichier,lignes_lues,lignes_vides_supprimees,lignes_sans_GPS_supprimees,lignes_ecrites,destination
0,delivery/delivery_cq.csv,931351,0,0,931351,data/processed/delivery/delivery_cq.csv
1,delivery/delivery_hz.csv,1861600,0,0,1861600,data/processed/delivery/delivery_hz.csv
2,delivery/delivery_jl.csv,31415,0,0,31415,data/processed/delivery/delivery_jl.csv
3,delivery/delivery_sh.csv,1483864,0,0,1483864,data/processed/delivery/delivery_sh.csv
4,delivery/delivery_yt.csv,206431,0,0,206431,data/processed/delivery/delivery_yt.csv
5,pickup/pickup_cq.csv,1172703,0,263196,909507,data/processed/pickup/pickup_cq.csv
6,pickup/pickup_hz.csv,2130456,0,647824,1482632,data/processed/pickup/pickup_hz.csv
7,pickup/pickup_jl.csv,261801,0,31337,230464,data/processed/pickup/pickup_jl.csv
8,pickup/pickup_sh.csv,1424406,0,382426,1041980,data/processed/pickup/pickup_sh.csv
9,pickup/pickup_yt.csv,1146781,0,178735,968046,data/processed/pickup/pickup_yt.csv


In [4]:
# Enregistre le tableau récapitulatif comme image PNG dans reports/.
import matplotlib.pyplot as plt

report_dir = project_root / "reports"
report_dir.mkdir(parents=True, exist_ok=True)
report_path = report_dir / "tableau_nettoyage.png"

report_table = summary[[
    "fichier",
    "lignes_lues",
    "lignes_vides_supprimees",
    "lignes_sans_GPS_supprimees",
    "lignes_ecrites",
]].copy()
report_table.columns = [
    "Fichier",
    "Lignes lues",
    "Lignes vides supprimées",
    "Lignes sans GPS supprimées",
    "Lignes écrites",
]

fig, ax = plt.subplots(figsize=(12, max(3, 0.48 * len(report_table) + 1)))
ax.axis("off")
ax.set_title("Bilan du nettoyage des données", fontsize=15, weight="bold", pad=18)
table = ax.table(
    cellText=report_table.values,
    colLabels=report_table.columns,
    cellLoc="center",
    colLoc="center",
    loc="center",
)
table.auto_set_font_size(False)
table.set_fontsize(9)
table.scale(1, 1.5)
for (row, col), cell in table.get_celld().items():
    if row == 0:
        cell.set_text_props(weight="bold", color="white")
        cell.set_facecolor("#28527a")
    elif row % 2 == 0:
        cell.set_facecolor("#edf3f8")
fig.tight_layout()
fig.savefig(report_path, dpi=200, bbox_inches="tight")
plt.close(fig)
print(f"Image du tableau enregistrée : {report_path}")

Image du tableau enregistrée : /home/yourix/Documents/PROJECTS/IOT/delivery-car/reports/tableau_nettoyage.png
